# 13 · Análisis de texto con PySpark

**Pregunta de clase.** ¿Cómo convertimos textos en números y agrupamos los que usan vocabulario parecido? Trabajaremos con titulares ficticios en español.

## 0 · Preparar el entorno

Elige un kernel de **Python 3.11 o 3.12** con **Java 17**. En Jupyter o VS Code usa el mismo entorno que en las prácticas anteriores. En Colab, comprueba la versión con `!python --version`. Si muestra Python 3.13, en **Entorno de ejecución → Cambiar tipo de entorno de ejecución** selecciona la versión **2026.07**, que usa Python 3.12. Si no tienes Java 17, ejecuta en una celda de Colab `!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless` y, en otra celda, `import os; os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"`.

Si faltan las bibliotecas, ejecuta en una celda `%pip install "pyspark>=3.5,<3.6" "numpy<2"` y reinicia el kernel si te lo pide. Puedes consultar la [guía del curso](../01_environment_pyspark.md). Ejecuta las celdas de arriba abajo. Los datos de ejemplo están dentro del notebook y el cálculo se hace en el equipo donde corre el kernel.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import CountVectorizer, IDF, Normalizer, RegexTokenizer, StopWordsRemover
from pyspark.ml.clustering import KMeans
from pyspark.sql.window import Window

spark = (SparkSession.builder.master('local[2]').appName('Analisis de texto')
         .config('spark.sql.shuffle.partitions', '4').getOrCreate())
print(f'Spark {spark.version}')

## 1 · Crear una tabla de textos

Cada fila contiene un titular corto. Incluimos tres temas para poder comprobar si los grupos tienen sentido. Spark recibe solo el identificador y el texto, sin etiquetas de tema.

In [ ]:
titulares = [
    'El equipo gana el partido de liga con dos goles',
    'El entrenador prepara el partido de liga del equipo',
    'El equipo marca tres goles en el partido de liga',
    'La liga publica el calendario de partidos del equipo',
    'Los goles del equipo deciden el partido de liga',
    'El partido de liga acaba con un gol del equipo',
    'El equipo busca goles para ganar el partido de liga',
    'La liga premia al equipo con más goles por partido',
    'La ciudad mejora la línea de autobús del barrio',
    'El autobús del barrio cambia de ruta por las obras',
    'La ciudad anuncia una nueva ruta de autobús',
    'El barrio recibe una parada de autobús en la ciudad',
    'La ruta de autobús conecta el barrio con la ciudad',
    'El autobús amplía el horario de la ruta del barrio',
    'La ciudad abre otra parada en la ruta de autobús',
    'El barrio pide un autobús para conectar con la ciudad',
    'El huerto escolar planta verduras para la cosecha',
    'La cosecha de verduras del huerto escolar mejora',
    'El huerto escolar prepara la tierra para las verduras',
    'El huerto escolar recoge verduras de la cosecha',
    'La escuela organiza la cosecha de verduras del huerto',
    'Las verduras de la cosecha crecen en el huerto escolar',
    'El huerto escolar celebra la primera cosecha de verduras',
    'La escuela riega las verduras del huerto para la cosecha',
]
documentos = spark.createDataFrame(list(enumerate(titulares, start=1)), ['documento', 'texto'])
documentos.show(5, truncate=False)

## 2 · Separar las palabras

Dividimos el texto por los caracteres que no son letras y pasamos las palabras a minúscula. Quitamos palabras frecuentes como «el» y «de», que suelen aportar poco para distinguir temas.

In [ ]:
tokenizador = RegexTokenizer(inputCol='texto', outputCol='palabras', pattern=r'\P{L}+')
palabras = tokenizador.transform(documentos)
filtro = StopWordsRemover(inputCol='palabras', outputCol='terminos',
                          stopWords=StopWordsRemover.loadDefaultStopWords('spanish'))
terminos = filtro.transform(palabras).filter(F.size('terminos') > 0)
terminos.select('documento', 'terminos').show(5, truncate=False)

## 3 · Convertir palabras en un vector

Primero contamos cuántas veces aparece cada término por documento. TF-IDF reduce el peso de términos que aparecen en casi todos los textos. Normalizamos la longitud del vector para que los titulares más largos tengan menos ventaja por contener más palabras.

In [ ]:
vocabulario = CountVectorizer(inputCol='terminos', outputCol='frecuencias').fit(terminos)
frecuencias = vocabulario.transform(terminos)
pesos = IDF(inputCol='frecuencias', outputCol='tfidf').fit(frecuencias).transform(frecuencias)
vectores = Normalizer(inputCol='tfidf', outputCol='variables').transform(pesos)
vectores.select('documento', 'variables').show(3, truncate=55)

## 4 · Formar grupos y ponerles nombre

Pedimos tres grupos para esta demostración. K-Means agrupa vectores parecidos y devuelve un número de grupo. Leemos los titulares y los términos frecuentes antes de decidir si cada grupo tiene una temática clara.

In [ ]:
modelo = KMeans(k=3, seed=42, featuresCol='variables', predictionCol='grupo').fit(vectores)
agrupados = modelo.transform(vectores)
agrupados.groupBy('grupo').count().orderBy('grupo').show()
agrupados.orderBy('grupo', 'documento').select('grupo', 'texto').show(len(titulares), truncate=False)

frecuencia = (agrupados.select('grupo', F.explode('terminos').alias('termino'))
              .groupBy('grupo', 'termino').count())
orden = Window.partitionBy('grupo').orderBy(F.desc('count'), 'termino')
(frecuencia.withColumn('puesto', F.row_number().over(orden))
 .filter(F.col('puesto') <= 4).orderBy('grupo', 'puesto').show())

**Interpretación.** El número de grupo es una etiqueta del algoritmo. Comprueba si términos como «liga», «autobús» y «huerto» aparecen en grupos distintos. En textos reales puede haber temas mezclados o grupos difíciles de interpretar.

## Cerrar la sesión

Liberamos los recursos de Spark al terminar este cuaderno.

In [ ]:
spark.stop()

## Navegación

[Volver al README](../README.md) · [Siguiente: Segmentación con K-Means](14_census_kmeans.ipynb)

Documentación de Spark ML. [K-Means](https://spark.apache.org/docs/3.5.6/ml-clustering.html) · [Preparación de variables](https://spark.apache.org/docs/3.5.6/ml-features.html)